<a href="https://colab.research.google.com/github/ekaratnida/Data-Engineer/blob/main/de/module_02_Ekarat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 04 — Workshop: การนำเข้า สำรวจ แปลง และตรวจสอบข้อมูลด้วย Spark/BigQuery

**ผู้สอน:** ผศ.ดร. เอกรัฐ รัฐกาญจน์

## ภาพรวม Workshop (1.5 ชม.)
- **ส่วนที่ 1:** ตั้งค่าและนำเข้าข้อมูลด้วย PySpark + BigQuery (30 นาที)
- **ส่วนที่ 2:** สำรวจและแปลงข้อมูล (Transformation) (35 นาที)
- **ส่วนที่ 3:** ตรวจสอบคุณภาพข้อมูล (Data Quality) (20 นาที)

### สิ่งที่ต้องเตรียม
- Google Colab (หรือ Jupyter Notebook)
- บัญชี Google Cloud (สำหรับ BigQuery) — หรือใช้ sample data

---
## ส่วนที่ 1: ตั้งค่าและนำเข้าข้อมูลด้วย PySpark + BigQuery

### ติดตั้ง PySpark ใน Colab

In [1]:
# สั่งติดตั้ง PySpark ใน Colab
#!pip install pyspark py4j

# ติดตั้ง BigQuery client
#!pip install google-cloud-bigquery

### ตรวจสอบเวอร์ชัน

In [1]:
import pyspark
print(f"PySpark version: {pyspark.__version__}")

PySpark version: 4.0.2


> **หมายเหตุ:** PySpark ทำงานบน Colab ได้ แต่จำกัดทรัพยากร — เหมาะสำหรับการเรียนรู้เท่านั้น

### เริ่มต้น SparkSession

In [2]:
from pyspark.sql import SparkSession

# สร้าง SparkSession
spark = SparkSession.builder \
    .appName("DataEng Workshop") \
    .config("spark.sql.repl.eagerEval.enabled", True) \
    .config("spark.sql.adaptive.enabled", True) \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

# ทดสอบว่าทำงาน
print("Spark version:", spark.version)

Spark version: 4.0.2


- **spark.sql.repl.eagerEval.enabled** — แสดง DataFrame โดยอัตโนมัติใน notebook
- **spark.sql.shuffle.partitions** — กำหนด partition สำหรับการ shuffle (ปรับตามขนาด cluster)

### การนำเข้าข้อมูล: CSV

In [8]:
# หรือโหลดจาก URL โดยตรง (Colab)
!wget -q https://raw.githubusercontent.com/ekaratnida/Data-Engineer/refs/heads/main/de/dataset/sales.csv

# อ่านไฟล์ CSV พร้อมกำหนด schema
df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("delimiter", ",") \
    .load("sample_data.csv")
df.show(5)

+--------------+------+-----------+-----+---+
|          name|amount|   category|price|qty|
+--------------+------+-----------+-----+---+
|      Notebook|  1200|Electronics| 1200|  1|
|Wireless Mouse|   450|Electronics|  150|  3|
|    Desk Chair|  2500|  Furniture| 2500|  1|
|    Coffee Mug|   320|    Kitchen|   80|  4|
|  Water Bottle|  1050|    Kitchen|  350|  3|
+--------------+------+-----------+-----+---+
only showing top 5 rows


In [14]:
from pyspark.sql.functions import col, format_string

# Create a DataFrame version of the high_value_items logic
# 1. Filter the DataFrame
df_high_value_df_version = df.filter(
    (col("category") == 'Electronics') & (col("amount") > 1000)
)

# 2. Map/transform the filtered DataFrame into the desired string format
high_value_items_df = df_high_value_df_version.select(
    format_string("Item: %s, Amount: %s, Category: %s", col("name"), col("amount"), col("category"))
).rdd.map(lambda x: x[0]).collect()

print("\n--- High Value Electronics Items (DataFrame version - first 5) ---")
for item in high_value_items_df[:5]:
    print(item)



--- High Value Electronics Items (DataFrame version - first 5) ---
Item: Notebook, Amount: 1200, Category: Electronics


In [13]:
# Convert DataFrame to RDD
df_rdd = df.rdd

print("--- Original RDD (first 5 elements) ---")
print(df_rdd.take(5))

# Example 1: Using map to extract specific columns (name, amount)
# Each row in the RDD is a Row object, you can access elements by index or name
mapped_rdd = df_rdd.map(lambda row: (row.name, row.amount))

print("\n--- Mapped RDD (name, amount - first 5 elements) ---")
for row in mapped_rdd.take(5):
    print(row)

# Example 2: Using filter to get items with amount > 1000
filtered_rdd = df_rdd.filter(lambda row: row.amount > 1000)

print("\n--- Filtered RDD (amount > 1000 - first 5 elements) ---")
for row in filtered_rdd.take(5):
  print(row)

# Example 3: Combining map and filter, then collecting results as a list
high_value_items = df_rdd \
    .filter(lambda row: row.category == 'Electronics' and row.amount > 1000) \
    .map(lambda row: f"Item: {row.name}, Amount: {row.amount}, Category: {row.category}") \
    .collect()

print("\n--- High Value Electronics Items (first 5) ---")
for item in high_value_items[:5]:
    print(item)

--- Original RDD (first 5 elements) ---
[Row(name='Notebook', amount=1200, category='Electronics', price=1200, qty=1), Row(name='Wireless Mouse', amount=450, category='Electronics', price=150, qty=3), Row(name='Desk Chair', amount=2500, category='Furniture', price=2500, qty=1), Row(name='Coffee Mug', amount=320, category='Kitchen', price=80, qty=4), Row(name='Water Bottle', amount=1050, category='Kitchen', price=350, qty=3)]

--- Mapped RDD (name, amount - first 5 elements) ---
('Notebook', 1200)
('Wireless Mouse', 450)
('Desk Chair', 2500)
('Coffee Mug', 320)
('Water Bottle', 1050)

--- Filtered RDD (amount > 1000 - first 5 elements) ---
Row(name='Notebook', amount=1200, category='Electronics', price=1200, qty=1)
Row(name='Desk Chair', amount=2500, category='Furniture', price=2500, qty=1)
Row(name='Water Bottle', amount=1050, category='Kitchen', price=350, qty=3)

--- High Value Electronics Items (first 5) ---
Item: Notebook, Amount: 1200, Category: Electronics


### การนำเข้าข้อมูล: Parquet

In [5]:
import os

# Define a temporary path for the Parquet file
parquet_output_path = "/tmp/sales_parquet"

# Create a dummy Parquet file from df_sales (assuming df_sales exists from previous cells)
# Using 'overwrite' mode to ensure it works if the cell is run multiple times
df_sales.write.mode("overwrite").parquet(parquet_output_path)
print(f"Dummy Parquet file created at: {parquet_output_path}")

# Parquet — รองรับ schema ในตัว, columnar, มี compression
df_parquet = spark.read \
    .format("parquet") \
    .load(parquet_output_path)

print("Content of df_parquet:")
df_parquet.show(5)

# อ่านทั้ง directory (which is what spark.write.parquet creates)
df_multi = spark.read.parquet(parquet_output_path)
print("Content of df_multi (reading from directory):")
df_multi.show(5)

Dummy Parquet file created at: /tmp/sales_parquet
Content of df_parquet:
+--------------+-----------+----------+---+----------+--------+-------------------+---------+
|transaction_id|customer_id|product_id|qty|unit_price|  amount|         order_date|   status|
+--------------+-----------+----------+---+----------+--------+-------------------+---------+
|             1|        169|         8|  6|   3789.04|22734.24|2023-03-29 08:16:00|completed|
|             2|         46|        32|  6|   2145.84|12875.04|2023-03-27 19:32:00|completed|
|             3|        160|        50|  1|   1409.87| 1409.87|2023-01-02 00:32:00|  pending|
|             4|        158|        33|  4|   2004.66| 8018.64|2023-04-13 01:37:00|completed|
|             5|        174|        50|  2|   1409.87| 2819.74|2023-04-16 22:43:00|completed|
+--------------+-----------+----------+---+----------+--------+-------------------+---------+
only showing top 5 rows
Content of df_multi (reading from directory):
+----------

### ข้อดีของ Parquet

| CSV | Parquet |
|---|---|
| Row-based | Columnar |
| ไม่มี schema | Schema embedded |
| ไม่บีบอัด | บีบอัด (Snappy, Gzip, Zstd) |
| อ่านช้า | อ่านเร็ว (predicate pushdown) |
| ไฟล์ใหญ่กว่า | ไฟล์เล็กกว่า 75-87% |

### การนำเข้าข้อมูล: JSON (Nested)

In [6]:
import json
import os

# Define the content for nested_data.json
json_content = {
    "user": {
        "name": "สมชาย",
        "age": 30,
        "address": {
            "city": "กรุงเทพ",
            "zip": "10100"
        }
    }
}

# Define the path for the JSON file
json_file_path = "nested_data.json"

# Write the JSON content to the file
with open(json_file_path, 'w', encoding='utf-8') as f:
    json.dump(json_content, f, ensure_ascii=False, indent=4)

print(f"'{json_file_path}' created successfully.")

# JSON — รองรับโครงสร้างซับซ้อน
df_json = spark.read \
    .format("json") \
    .option("multiline", "true") \
    .load(json_file_path)

# แสดง schema ของ JSON
df_json.printSchema()

# เข้าถึง nested field
df_json.select("user.name", "user.address.city").show()

'nested_data.json' created successfully.
root
 |-- user: struct (nullable = true)
 |    |-- address: struct (nullable = true)
 |    |    |-- city: string (nullable = true)
 |    |    |-- zip: string (nullable = true)
 |    |-- age: long (nullable = true)
 |    |-- name: string (nullable = true)

+-----+-------+
| name|   city|
+-----+-------+
|สมชาย|กรุงเทพ|
+-----+-------+



ตัวอย่างข้อมูล JSON:
```json
{"user": {"name": "สมชาย", "age": 30, "address": {"city": "กรุงเทพ", "zip": "10100"}}}
```

## BigQuery

https://docs.google.com/presentation/d/1PddRr1WBlpKB4RMzRDgqzrk7q5C97fJU1h9sYWWJDb8/edit?slide=id.g3e4e1b92149_0_396&pli=1#slide=id.g3e4e1b92149_0_396


In [8]:
from google.colab import auth
from google.cloud import bigquery

# 1. Authenticate with Google Cloud
auth.authenticate_user()

# 2. Set your Project ID (Required for BigQuery Sandbox/Billing)
# Please enter your project ID here:
project_id = input("Please enter your Google Cloud Project ID: ")

# 3. Initialize the client using YOUR project as the billing project
client = bigquery.Client(project=project_id)

# Query ข้อมูลจาก BigQuery
query = """
    SELECT pickup_datetime, dropoff_datetime,
           passenger_count, trip_distance, fare_amount
    FROM `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2022`
    WHERE fare_amount > 0
    LIMIT 10000
"""

df = client.query(query).to_dataframe()
print(f"Loaded {len(df)} rows")
display(df.head())

Please enter your Google Cloud Project ID: vaulted-art-485212-r1
Loaded 10000 rows


,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,fare_amount
0,2022-04-07 07:35:00+00:00,2022-04-07 07:49:00+00:00,<NA>,1.480000000,1.000000000
1,2022-04-26 23:03:00+00:00,2022-04-26 23:40:00+00:00,<NA>,8.150000000,30.720000000
2,2022-04-10 15:08:22+00:00,2022-04-10 15:26:48+00:00,<NA>,8.210000000,35.840000000
3,2022-04-10 23:44:00+00:00,2022-04-10 23:55:00+00:00,<NA>,1.990000000,10.240000000
4,2022-04-05 09:17:00+00:00,2022-04-05 09:29:00+00:00,<NA>,1.690000000,10.240000000


### สำรวจข้อมูลเบื้องต้น

In [9]:
from pyspark.sql.functions import count, when, col
from pyspark.sql import SparkSession

# Fix for Spark 4.0.2 Internal Error: Re-asserting the active session
spark = SparkSession.builder.getOrCreate()

# ดู schema และตัวอย่างข้อมูล
df_sales.printSchema()
df_sales.show(5, truncate=False)

# สถิติเบื้องต้นของ numeric columns
df_sales.describe().show()

# ดูจำนวน record
print(f"Total rows: {df_sales.count()}")
print(f"Total columns: {len(df_sales.columns)}")
print(f"Column names: {df_sales.columns}")

# ดูค่า NULL
# Fixed: Added explicit imports for count, when, and col
df_sales.select([count(when(col(c).isNull(), c)).alias(c)
           for c in df_sales.columns]).show()

root
 |-- transaction_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- qty: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- amount: double (nullable = true)
 |-- order_date: timestamp (nullable = true)
 |-- status: string (nullable = true)

+--------------+-----------+----------+---+----------+--------+-------------------+---------+
|transaction_id|customer_id|product_id|qty|unit_price|amount  |order_date         |status   |
+--------------+-----------+----------+---+----------+--------+-------------------+---------+
|1             |169        |8         |6  |3789.04   |22734.24|2023-03-29 08:16:00|completed|
|2             |46         |32        |6  |2145.84   |12875.04|2023-03-27 19:32:00|completed|
|3             |160        |50        |1  |1409.87   |1409.87 |2023-01-02 00:32:00|pending  |
|4             |158        |33        |4  |2004.66   |8018.64 |2023-04-13 01:37:00|completed|
|5 

### สรุปส่วนที่ 1

- **SparkSession** — entry point สำหรับ Spark
- **Data Sources** — CSV, Parquet, JSON, BigQuery
- **Parquet** เป็น format ที่แนะนำ (columnar, compressed, schema)
- **BigQuery** — query direct หรือใช้งานผ่าน Spark connector
- **Basic Exploration** — `printSchema()`, `show()`, `describe()`, `count()`

---
## ส่วนที่ 2: การแปลงข้อมูล (Transformation) และสำรวจเชิงลึก

### Column Operations

In [10]:
df = df_sales
from pyspark.sql.functions import col, lit, when, expr

# 1. เลือก column (เปลี่ยนจาก 'date' เป็น 'order_date')
print("--- Selecting Columns ---")
df.select("order_date", "amount").show(5)

# 2. สร้าง column ใหม่ (เปลี่ยนจาก 'price' เป็น 'unit_price')
print("--- Creating New Columns ---")
df.withColumn("total_amount_calc", col("unit_price") * col("qty")) \
  .withColumn("tax", col("amount") * 0.07) \
  .select("transaction_id", "amount", "tax") \
  .show(5)

# 3. เปลี่ยนชื่อ column
print("--- Renaming Columns ---")
df_renamed = df.withColumnRenamed("status", "order_status")
df_renamed.select("transaction_id", "order_status").show(5)

# 4. กรองตามเงื่อนไข
print("--- Filtering Data ---")
df.filter(col("amount") > 10000).show(5)
df.where("amount > 1000 AND status = 'completed'").show(5)

--- Selecting Columns ---
+-------------------+--------+
|         order_date|  amount|
+-------------------+--------+
|2023-03-29 08:16:00|22734.24|
|2023-03-27 19:32:00|12875.04|
|2023-01-02 00:32:00| 1409.87|
|2023-04-13 01:37:00| 8018.64|
|2023-04-16 22:43:00| 2819.74|
+-------------------+--------+
only showing top 5 rows
--- Creating New Columns ---
+--------------+--------+------------------+
|transaction_id|  amount|               tax|
+--------------+--------+------------------+
|             1|22734.24|1591.3968000000002|
|             2|12875.04| 901.2528000000001|
|             3| 1409.87|           98.6909|
|             4| 8018.64| 561.3048000000001|
|             5| 2819.74|          197.3818|
+--------------+--------+------------------+
only showing top 5 rows
--- Renaming Columns ---
+--------------+------------+
|transaction_id|order_status|
+--------------+------------+
|             1|   completed|
|             2|   completed|
|             3|     pending|
|       

### ตัวอย่าง Real-World: E-commerce Data

In [11]:
# สมมติข้อมูลคำสั่งซื้อ
orders = spark.createDataFrame([
    ("C001", "2024-01-15", "A", 1200, 2),
    ("C002", "2024-01-15", "B", 500, 1),
    ("C003", "2024-01-16", "A", 300, 5),
    ("C004", "2024-01-16", "C", 2500, 3),
    ("C001", "2024-01-17", "B", 800, 1),
], schema=["customer_id", "order_date", "category", "amount", "qty"])

# เพิ่ม column: total_revenue
orders = orders.withColumn("revenue",
    col("amount") * col("qty"))

# เพิ่ม column หมวดราคา
orders = orders.withColumn("price_tier",
    when(col("amount") >= 1000, "High")
    .when(col("amount") >= 500, "Medium")
    .otherwise("Low"))

orders.show()

+-----------+----------+--------+------+---+-------+----------+
|customer_id|order_date|category|amount|qty|revenue|price_tier|
+-----------+----------+--------+------+---+-------+----------+
|       C001|2024-01-15|       A|  1200|  2|   2400|      High|
|       C002|2024-01-15|       B|   500|  1|    500|    Medium|
|       C003|2024-01-16|       A|   300|  5|   1500|       Low|
|       C004|2024-01-16|       C|  2500|  3|   7500|      High|
|       C001|2024-01-17|       B|   800|  1|    800|    Medium|
+-----------+----------+--------+------+---+-------+----------+



### Aggregation

In [12]:
from pyspark.sql.functions import sum, avg, count, max, min, stddev, round

# Group by เดียว
orders.groupBy("category") \
    .agg(
        count("*").alias("order_count"),
        sum("revenue").alias("total_revenue"),
        avg("amount").alias("avg_amount"),
        max("amount").alias("max_amount"),
        round(avg("qty"), 2).alias("avg_qty")
    ) \
    .orderBy(col("total_revenue").desc()) \
    .show()

+--------+-----------+-------------+----------+----------+-------+
|category|order_count|total_revenue|avg_amount|max_amount|avg_qty|
+--------+-----------+-------------+----------+----------+-------+
|       C|          1|         7500|    2500.0|      2500|    3.0|
|       A|          2|         3900|     750.0|      1200|    3.5|
|       B|          2|         1300|     650.0|       800|    1.0|
+--------+-----------+-------------+----------+----------+-------+



### Multiple Aggregations

In [13]:
# Group by หลาย dimensions
orders.groupBy("customer_id", "category") \
    .agg(
        sum("revenue").alias("total_spent"),
        count("*").alias("num_orders"),
        avg("qty").alias("avg_qty")
    ) \
    .orderBy(col("total_spent").desc()) \
    .show()

# Pivot table
orders.groupBy("customer_id") \
    .pivot("category") \
    .agg(sum("revenue")) \
    .fillna(0) \
    .show()

+-----------+--------+-----------+----------+-------+
|customer_id|category|total_spent|num_orders|avg_qty|
+-----------+--------+-----------+----------+-------+
|       C004|       C|       7500|         1|    3.0|
|       C001|       A|       2400|         1|    2.0|
|       C003|       A|       1500|         1|    5.0|
|       C001|       B|        800|         1|    1.0|
|       C002|       B|        500|         1|    1.0|
+-----------+--------+-----------+----------+-------+

+-----------+----+---+----+
|customer_id|   A|  B|   C|
+-----------+----+---+----+
|       C001|2400|800|   0|
|       C002|   0|500|   0|
|       C004|   0|  0|7500|
|       C003|1500|  0|   0|
+-----------+----+---+----+



### Joins

In [14]:
# ข้อมูลลูกค้า
customers = spark.createDataFrame([
    ("C001", "สมชาย", "Gold"),
    ("C002", "สมหญิง", "Silver"),
    ("C003", "สมศักดิ์", "Platinum"),
    ("C005", "สมปอง", "Bronze"),
], schema=["customer_id", "name", "tier"])

# Inner Join
orders.join(customers, on="customer_id", how="inner").show()

# Left Join (เก็บข้อมูล orders ทั้งหมด)
orders.join(customers, on="customer_id", how="left").show()

# Anti Join (ลูกค้าที่ไม่มี order)
customers.join(orders, on="customer_id", how="left_anti").show()

+-----------+----------+--------+------+---+-------+----------+--------+--------+
|customer_id|order_date|category|amount|qty|revenue|price_tier|    name|    tier|
+-----------+----------+--------+------+---+-------+----------+--------+--------+
|       C001|2024-01-15|       A|  1200|  2|   2400|      High|   สมชาย|    Gold|
|       C002|2024-01-15|       B|   500|  1|    500|    Medium|  สมหญิง|  Silver|
|       C001|2024-01-17|       B|   800|  1|    800|    Medium|   สมชาย|    Gold|
|       C003|2024-01-16|       A|   300|  5|   1500|       Low|สมศักดิ์|Platinum|
+-----------+----------+--------+------+---+-------+----------+--------+--------+

+-----------+----------+--------+------+---+-------+----------+--------+--------+
|customer_id|order_date|category|amount|qty|revenue|price_tier|    name|    tier|
+-----------+----------+--------+------+---+-------+----------+--------+--------+
|       C001|2024-01-15|       A|  1200|  2|   2400|      High|   สมชาย|    Gold|
|       C002|20

### Window Functions

In [15]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, dense_rank, row_number, lag, lead

# จัดอันดับยอดขายตามหมวดสินค้า
window_spec = Window.partitionBy("category") \
    .orderBy(col("revenue").desc())

orders.withColumn("rank", rank().over(window_spec)) \
    .withColumn("dense_rank", dense_rank().over(window_spec)) \
    .withColumn("row_num", row_number().over(window_spec)) \
    .filter(col("rank") <= 2) \
    .show()

# Running total
window_rolling = Window.partitionBy("customer_id") \
    .orderBy("order_date") \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

orders.withColumn("running_total",
    sum("revenue").over(window_rolling)) \
    .show()

+-----------+----------+--------+------+---+-------+----------+----+----------+-------+
|customer_id|order_date|category|amount|qty|revenue|price_tier|rank|dense_rank|row_num|
+-----------+----------+--------+------+---+-------+----------+----+----------+-------+
|       C001|2024-01-15|       A|  1200|  2|   2400|      High|   1|         1|      1|
|       C003|2024-01-16|       A|   300|  5|   1500|       Low|   2|         2|      2|
|       C001|2024-01-17|       B|   800|  1|    800|    Medium|   1|         1|      1|
|       C002|2024-01-15|       B|   500|  1|    500|    Medium|   2|         2|      2|
|       C004|2024-01-16|       C|  2500|  3|   7500|      High|   1|         1|      1|
+-----------+----------+--------+------+---+-------+----------+----+----------+-------+

+-----------+----------+--------+------+---+-------+----------+-------------+
|customer_id|order_date|category|amount|qty|revenue|price_tier|running_total|
+-----------+----------+--------+------+---+-------

### Using SQL with Spark

In [16]:
# สร้าง Temp View
orders.createOrReplaceTempView("orders")
customers.createOrReplaceTempView("customers")

# Query ด้วย SQL
spark.sql("""
    SELECT  c.customer_id,
            c.name,
            c.tier,
            COUNT(o.customer_id) AS order_count,
            SUM(o.revenue) AS total_revenue
    FROM    customers c
    LEFT JOIN orders o ON c.customer_id = o.customer_id
    GROUP BY c.customer_id, c.name, c.tier
    HAVING total_revenue > 1000
    ORDER BY total_revenue DESC
""").show()

+-----------+--------+--------+-----------+-------------+
|customer_id|    name|    tier|order_count|total_revenue|
+-----------+--------+--------+-----------+-------------+
|       C001|   สมชาย|    Gold|          2|         3200|
|       C003|สมศักดิ์|Platinum|          1|         1500|
+-----------+--------+--------+-----------+-------------+



### UDF: User Defined Functions

In [17]:
from pyspark.sql.functions import udf, pandas_udf, col
from pyspark.sql.types import StringType
import re

# 1. สร้างข้อมูลตัวอย่างที่มี Email เพื่อทดสอบ
user_data = spark.createDataFrame([
    ("somchai@gmail.com", "Somchai"),
    ("somying.work@company.com", "Somying"),
    ("ton_data@outlook.com", "Ton"),
    (None, "Unknown")
], schema=["email", "name"])

# 2. กำหนด UDF ด้วย decorator
@udf(returnType=StringType())
def mask_email(email):
    """ซ่อน email บางส่วน (Data Masking)"""
    if email is None:
        return None
    parts = email.split("@")
    if len(parts) == 2:
        name, domain = parts
        if len(name) <= 2:
            return f"{name}***@{domain}"
        masked = name[:2] + "*" * (len(name) - 2)
        return f"{masked}@{domain}"
    return email

# 3. ใช้งาน UDF บน DataFrame ใหม่
print("--- Standard UDF Output ---")
user_data.withColumn("masked_email", mask_email(col("email"))).show()

# 4. Pandas UDF (vectorized) — แก้ไข regex ให้รองรับ raw string
@pandas_udf(returnType=StringType())
def mask_email_pd(email_series):
    # ซ่อนตัวอักษรตั้งแต่ตำแหน่งที่ 3 จนถึงก่อนเครื่องหมาย @
    return email_series.str.replace(r"(?<=.{2}).(?=.*@)", "*", regex=True)

print("--- Pandas UDF Output ---")
user_data.withColumn("masked_email_pd", mask_email_pd(col("email"))).show()

--- Standard UDF Output ---
+--------------------+-------+--------------------+
|               email|   name|        masked_email|
+--------------------+-------+--------------------+
|   somchai@gmail.com|Somchai|   so*****@gmail.com|
|somying.work@comp...|Somying|so**********@comp...|
|ton_data@outlook.com|    Ton|to******@outlook.com|
|                NULL|Unknown|                NULL|
+--------------------+-------+--------------------+

--- Pandas UDF Output ---
+--------------------+-------+--------------------+
|               email|   name|     masked_email_pd|
+--------------------+-------+--------------------+
|   somchai@gmail.com|Somchai|   so*****@gmail.com|
|somying.work@comp...|Somying|so**********@comp...|
|ton_data@outlook.com|    Ton|to******@outlook.com|
|                NULL|Unknown|                NULL|
+--------------------+-------+--------------------+



### การจัดการ Missing Values

In [18]:
from pyspark.sql.functions import coalesce, when, isnan, count, avg, col

# 1. ดูจำนวน NULL ในแต่ละ column (Fixed for Spark 4.0.2 type safety)
# We check the data type to avoid calling isnan() on non-numeric columns like TIMESTAMP
print("--- Missing Values Check ---")
df.select([
    count(when(
        col(c).isNull() |
        (isnan(col(c)) if df.schema[c].dataType.simpleString() in ["double", "float"] else lit(False)),
        c
    )).alias(c) for c in df.columns
]).show()

# 2. ลบแถวที่มี NULL
df_clean = df.na.drop()  # ทุก column
df_clean = df.na.drop(subset=["customer_id", "amount"])

# 3. เติมค่า NULL
df_fill = df.na.fill(value=0)                                          # ตัวเลข
df_fill = df.na.fill(value="Unknown", subset=["status"])               # ข้อความ
df_fill = df.na.fill({"amount": 0, "status": "Unknown"})                # mix

# 4. เติมด้วยค่าเฉลี่ย
mean_amount = df.select(avg("amount")).collect()[0][0]
df_fill = df.na.fill({"amount": mean_amount})
print(f"Filled missing amounts with mean: {mean_amount:.2f}")

--- Missing Values Check ---
+--------------+-----------+----------+---+----------+------+----------+------+
|transaction_id|customer_id|product_id|qty|unit_price|amount|order_date|status|
+--------------+-----------+----------+---+----------+------+----------+------+
|             0|          0|         0|  0|         0|     0|         0|     0|
+--------------+-----------+----------+---+----------+------+----------+------+

Filled missing amounts with mean: 13190.01


### การจัดการ Duplicates

In [19]:
# ตรวจสอบ duplicate
from pyspark.sql.functions import count as spark_count, col

# ตรวจสอบความซ้ำซ้อนโดยใช้ customer_id และ order_date
df.groupBy("customer_id", "order_date") \
    .agg(spark_count("*").alias("cnt")) \
    .filter(col("cnt") > 1) \
    .show()

# ลบ duplicate (เก็บ record แรก)
# แก้ไข: เปลี่ยนจาก order_id เป็น transaction_id หรือ columns ที่มีอยู่ใน schema จริง
df_dedup = df.dropDuplicates()
df_dedup = df.dropDuplicates(subset=["transaction_id"])

# แสดงจำนวนก่อน-หลัง
print(f"Before: {df.count()} rows")
print(f"After:  {df_dedup.count()} rows")
print(f"Removed: {df.count() - df_dedup.count()} duplicates")

+-----------+----------+---+
|customer_id|order_date|cnt|
+-----------+----------+---+
+-----------+----------+---+

Before: 5000 rows
After:  5000 rows
Removed: 0 duplicates


### สรุปส่วนที่ 2

| Operation | Function | ตัวอย่าง |
|---|---|---|
| **Column** | `select`, `withColumn`, `when` | เพิ่ม/เลือก/เปลี่ยน column |
| **Filter** | `filter`, `where` | กรองข้อมูลตามเงื่อนไข |
| **Aggregate** | `groupBy`, `agg` | รวมข้อมูล, หาผลรวม |
| **Join** | `join` | เชื่อมตาราง (inner, left, anti) |
| **Window** | `rank`, `lag`, `sum over` | จัดอันดับ, running total |
| **UDF** | `udf`, `pandas_udf` | ฟังก์ชันกำหนดเอง |
| **Clean** | `na.drop`, `na.fill`, `dropDuplicates` | จัดการ NULL/duplicate |

---
## ส่วนที่ 3: ตรวจสอบคุณภาพข้อมูล (Data Quality)

### ทำไม Data Quality Checks ถึงสำคัญ

"Garbage In, Garbage Out" (GIGO)
- ข้อมูลที่มีคุณภาพต่ำ → ผลลัพธ์ที่ผิดพลาด
- ตรวจสอบคุณภาพตั้งแต่ **ต้น pipeline** เพื่อป้องกันปัญหา

### องค์ประกอบของ Data Quality Check
1. Schema Validation
2. Null / Missing Value Check
3. Uniqueness / Duplicate Check
4. Range / Outlier Detection
5. Referential Integrity
6. Custom Business Rules

### 1. Schema Validation

In [25]:
# ตรวจสอบ schema ที่คาดหวัง
expected_schema = {
    "customer_id": "string",
    "order_date": "string",
    "category": "string",
    "amount": "int",
    "qty": "int"
}

def check_schema(df, expected_schema):
    """ตรวจสอบ schema ว่าตรงตามที่คาดหวัง"""
    actual_schema = {f.name: f.dataType.simpleString()
                     for f in df.schema.fields}

    print("=== Schema Validation ===")
    for col_name, expected_type in expected_schema.items():
        actual_type = actual_schema.get(col_name)
        if actual_type is None:
            print(f"Missing column: {col_name}")
        elif actual_type != expected_type:
            print(f" Type mismatch: {col_name} "
                  f"(expected {expected_type}, got {actual_type})")
        else:
            print(f"{col_name}: {actual_type}")
    return actual_schema

check_schema(orders, expected_schema)

=== Schema Validation ===
customer_id: string
order_date: string
category: string
 Type mismatch: amount (expected int, got bigint)
 Type mismatch: qty (expected int, got bigint)


{'customer_id': 'string',
 'order_date': 'string',
 'category': 'string',
 'amount': 'bigint',
 'qty': 'bigint',
 'revenue': 'bigint',
 'price_tier': 'string'}

### 2. Null / Missing Value Check

In [26]:
def check_nulls(df, threshold=0.05):
    """ตรวจสอบ NULL value ในแต่ละ column"""
    total = df.count()
    print(f"Total rows: {total}")
    print("=== NULL Check ===")

    result = df.select([
        count(when(col(c).isNull(), c)).alias(c)
        for c in df.columns
    ]).collect()[0]

    errors = []
    for c in df.columns:
        null_count = result[c]
        null_ratio = null_count / total if total > 0 else 0

        if null_ratio > threshold:
            errors.append(f"{c}: {null_count} NULLs ({null_ratio:.1%})")
            print(f"{c}: {null_count} NULLs ({null_ratio:.1%}) — OVER threshold")
        elif null_count > 0:
            print(f"{c}: {null_count} NULLs ({null_ratio:.1%})")
        else:
            print(f"{c}: no NULLs")

    return errors

errors = check_nulls(orders, threshold=0.05)

Total rows: 5
=== NULL Check ===
customer_id: no NULLs
order_date: no NULLs
category: no NULLs
amount: no NULLs
qty: no NULLs
revenue: no NULLs
price_tier: no NULLs


### 3. Uniqueness / Duplicate Check

In [27]:
def check_uniqueness(df, primary_keys):
    """ตรวจสอบความซ้ำซ้อนของ primary key"""
    print(f"=== Uniqueness Check ===")
    print(f"Primary keys: {primary_keys}")

    total = df.select(primary_keys).distinct().count()
    dup = df.count() - total

    if dup > 0:
        print(f"Found {dup} duplicate records ({dup} rows)")
        # แสดง record ที่ซ้ำ
        df.groupBy(primary_keys) \
            .agg(count("*").alias("cnt")) \
            .filter(col("cnt") > 1) \
            .show()
    else:
        print(f"No duplicates found ({total} unique)")

# ตรวจสอบ customer_id + order_date ไม่ซ้ำ
check_uniqueness(orders, ["customer_id", "order_date"])

=== Uniqueness Check ===
Primary keys: ['customer_id', 'order_date']
No duplicates found (5 unique)


### 4. Range / Outlier Detection

In [28]:
def check_ranges(df, rules):
    """ตรวจสอบ range ของข้อมูล"""
    print("=== Range Validation ===")

    for col_name, min_val, max_val in rules:
        violators = df.filter(
            (col(col_name) < min_val) | (col(col_name) > max_val)
        ).count()

        if violators > 0:
            print(f"{col_name}: {violators} rows outside [{min_val}, {max_val}]")
        else:
            print(f"{col_name}: all within [{min_val}, {max_val}]")

# กำหนด range rules
range_rules = [
    ("amount", 0, 1000000),    # ยอดขาย 0 - 1,000,000
    ("qty", 1, 1000),          # จำนวนซื้อ 1 - 1000
    ("revenue", 0, 10000000)   # รายได้รวม 0 - 10,000,000
]

check_ranges(orders, range_rules)

=== Range Validation ===
amount: all within [0, 1000000]
qty: all within [1, 1000]
revenue: all within [0, 10000000]


### 5. Automated Data Quality Pipeline

In [31]:
from pyspark.sql.functions import count

class DataQualityChecker:
    """Data Quality Checker — ตรวจสอบคุณภาพข้อมูลอัตโนมัติ"""

    def __init__(self, df, name="Unnamed"):
        self.df = df
        self.name = name
        self.total = df.count()
        self.results = []

    def check_completeness(self, required_cols):
        """ตรวจสอบว่า column ที่จำเป็นมีข้อมูลครบ"""
        for c in required_cols:
            nulls = self.df.filter(col(c).isNull()).count()
            passed = nulls == 0
            self.results.append({
                "check": f"completeness.{c}",
                "passed": passed,
                "detail": f"{nulls}/{self.total} NULLs"
            })

    def check_unique(self, keys):
        """ตรวจสอบ uniqueness"""
        distinct = self.df.select(keys).distinct().count()
        passed = distinct == self.total
        self.results.append({
            "check": f"unique.{'_'.join(keys)}",
            "passed": passed,
            "detail": f"{distinct}/{self.total} unique"
        })

    def run(self):
        """รัน checks ทั้งหมดและสร้าง report"""
        print(f"\n{'='*50}")
        print(f"DQ Report: {self.name}")
        print(f"{'='*50}")
        print(f"Total rows: {self.total}")

        all_passed = True
        for r in self.results:
            icon = "Pass" if r["passed"] else "Fail"
            print(f"  {icon} {r['check']}: {r['detail']}")
            if not r["passed"]:
                all_passed = False

        print(f"\nResult: {'PASSED' if all_passed else 'FAILED'}")
        return all_passed

# ใช้งาน
checker = DataQualityChecker(orders, "Sales Orders")
checker.check_completeness(["customer_id", "amount", "qty"])
checker.check_unique(["customer_id", "order_date"])
# เพิ่ม check อื่นๆ...
checker.run()


DQ Report: Sales Orders
Total rows: 5
  Pass completeness.customer_id: 0/5 NULLs
  Pass completeness.amount: 0/5 NULLs
  Pass completeness.qty: 0/5 NULLs
  Pass unique.customer_id_order_date: 5/5 unique

Result: PASSED


True

### แนวปฏิบัติ Data Quality ที่ดี

| หลักการ | คำอธิบาย |
|---|---|
| **Check Early** | ตรวจสอบตั้งแต่ ingestion / staging |
| **Fail Fast** | ถ้าข้อมูลไม่ผ่าน quality → หยุด pipeline |
| **Alert on Failure** | แจ้งเตือนทีมทันที |
| **Log Everything** | บันทึกผล quality check ทุกครั้ง |
| **Define SLAs** | กำหนดเป้าหมายคุณภาพที่ชัดเจน |
| **Automate** | quality check ต้องเป็น automated (ไม่ manual) |
| **Ownership** | ทุก data set ต้องมีเจ้าของ |

### สรุป Module 04 Workshop

| ส่วน | หัวข้อ | เทคโนโลยี |
|---|---|---|
| **นำเข้าและสำรวจ** | CSV, Parquet, JSON, BigQuery | PySpark, google-cloud-bigquery |
| **แปลงข้อมูล** | Column ops, Agg, Join, Window, UDF | PySpark SQL |
| **ตรวจสอบคุณภาพ** | Schema, Null, Unique, Range | Custom DQ Class |

### ขั้นตอนถัดไป
- นำโค้ด DQ ไปปรับใช้ใน Airflow pipeline
- เพิ่ม Great Expectations สำหรับ enterprise DQ
- ใช้ Spark Structured Streaming สำหรับ real-time data